# NB1 · Kiểm tra kế toán → Tính chỉ tiêu → Verify & QA (Bước 1–3)
**FinReport Intelligence · Festival Chuyển đổi số 2026 · Bảng C · Case 3**

| | |
|---|---|
| **Đọc từ** | `03_outputs/step0/master_panel.csv` (kết quả NB0) · Data Pack `01_datapack/` (để verify độc lập và chấm QA) |
| **Ghi ra** | `03_outputs/step1_3/` |
| **Thư viện** | `lib/finreport_common.py` · `lib/finreport_metrics.py` · `lib/finreport_qa.py` |

| Bước | Câu hỏi trả lời | Kết quả |
|---|---|---|
| **1. Kiểm tra logic kế toán** | Con số bên trong các báo cáo có nhất quán về kế toán không? | 32 luật × 60 dòng, cờ dòng nhạy cảm |
| **2. Tính chỉ tiêu** | 18 chỉ tiêu tài chính của mỗi DN-năm là bao nhiêu? | Bảng 60 × 18, kèm công thức, nguồn, cờ diễn giải |
| **3. Verify & QA** | Phép tính có đúng không? Trả lời đúng câu hỏi của BTC không? | Verify kép, 30 câu QA so answer key, 300 câu tổng quát hóa, kiểm thử đột biến |

## 0. Kết nối Drive, nạp thư viện, kiểm tra kết quả NB0

In [ ]:
import os, sys, importlib, time, hashlib, dataclasses
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/FinReport_Intelligence"
except ImportError:
    PROJECT = os.environ.get("FINREPORT_PROJECT", "FinReport_Intelligence")

sys.path.insert(0, PROJECT + "/lib")
import finreport_common, finreport_metrics, finreport_qa
for mod in (finreport_common, finreport_metrics, finreport_qa):     # luôn dùng bản mới nhất trong lib/
    importlib.reload(mod)
from finreport_common import *
import finreport_metrics as fm
import finreport_qa as qa
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)

cfg = setup(PROJECT)
STEP = "step1_3"

In [ ]:
# Truy vết nguồn gốc: master_panel phải đến từ một lần chạy NB0 hợp lệ, trên đúng bộ dữ liệu hiện tại
master = read_output("step0", "master_panel.csv")
audit = read_audit()
s0 = audit[audit["step"] == "step0_ingest"].tail(1)
assert len(s0), "Chưa có bản ghi NB0 trong audit log – hãy chạy NB0_Ingest trước"
s0 = s0.iloc[0]
current_fp = hashlib.sha256("".join(sha256_of(locate_pack_file(k)[0]) for k in sorted(CORE_FILES)).encode()).hexdigest()

print(f"NB0 gần nhất : {s0['run_id']} · {s0['verdict']}")
print(f"Fingerprint  : NB0 {s0['data_fingerprint'][:16]}… | hiện tại {current_fp[:16]}…")
assert not str(s0["verdict"]).startswith("FAIL"), "NB0 gần nhất FAIL – không được tính tiếp"
assert s0["data_fingerprint"] == current_fp, "Dữ liệu đã thay đổi sau lần chạy NB0 – hãy chạy lại NB0"
print(f"master_panel : {master.shape[0]} dòng × {master.shape[1]} cột  →  OK, dùng được")
PARENT_RUN = s0["run_id"]

---
## BƯỚC 1 · Kiểm tra logic kế toán

Bước 0 kiểm tra **file**; Bước 1 kiểm tra **con số bên trong** có nhất quán theo nguyên tắc kế toán.
Mỗi luật được áp cho từng company-year; dung sai 0,5 triệu VND (số liệu là số nguyên nên lệch > 0,5 là sai thật).

| Nhóm | Số luật | Nội dung |
|---|---:|---|
| A. Kết quả kinh doanh | 5 | Các tầng lợi nhuận cộng trừ đúng; thuế không âm |
| B. Bảng cân đối | 7 | Tổng = Σ thành phần; **Tài sản = Nợ + Vốn chủ** |
| C. Dòng tiền trong năm | 6 | CFO/CFI/CFF = Σ khoản mục; tiền đầu + biến động = tiền cuối = tiền trên BCĐKT |
| D. Liên kết hai năm | 14 | Biến động trên BCĐKT khớp dòng tiền (phải thu, tồn kho, nợ vay, LN giữ lại, TSCĐ…). **Không áp dụng cho 2021** → `N/A` |

In [ ]:
display(fm.rule_catalog())

In [ ]:
t0 = time.perf_counter()
RUN_S1 = new_run_id("S1")
d, p = fm.prepare(master)
acc_checks = fm.run_accounting_checks(d, p)
acc_summary = fm.summarize_checks(acc_checks)
flags = fm.row_flags(d, p)
ms_s1 = (time.perf_counter() - t0) * 1000

n = acc_checks["status"].value_counts().to_dict()
S1_VERDICT = "FAIL" if n.get("FAIL") else "PASS"
print(f"Run {RUN_S1}: {S1_VERDICT} · {n} · {ms_s1:.0f} ms")
display(acc_summary)

In [ ]:
# Các phép kiểm tra không đạt (nếu có) – kèm vế trái, vế phải, độ lệch để truy vết
fails = acc_checks[acc_checks["status"] == "FAIL"]
print("Không có phép kiểm tra nào FAIL." if fails.empty else f"{len(fails)} phép FAIL:")
if len(fails):
    display(fails[["rule_id", "company_id", "fiscal_year", "detail", "evidence"]])

# Ví dụ chi tiết một luật: Tài sản = Nợ + Vốn chủ cho C012
display(acc_checks.query("rule_id == 'BS06' and company_id == 'C012'")[["rule_id", "company_id", "fiscal_year", "status", "detail"]])

### Cờ dòng nhạy cảm
Không phải lỗi dữ liệu, mà là các dòng mà **một số tỷ số cần diễn giải thận trọng** ở bước cảnh báo
(ví dụ CFO/LNST khi LNST ≤ 0: cả tử và mẫu cùng âm cho ra số dương "đẹp" nhưng vô nghĩa).

In [ ]:
display(flags[flags["row_flags"].str.contains("NET_LOSS|NEGATIVE_CFO|OPERATING_LOSS|OP_BELOW_INTEREST|NEGATIVE_EQUITY")])

save_output(acc_checks, STEP, "accounting_checks.csv")
save_output(acc_summary, STEP, "accounting_rule_summary.csv")
save_output(flags, STEP, "row_flags.csv")
append_audit({"run_id": RUN_S1, "step": "step1_accounting", "parent_run": PARENT_RUN, "verdict": S1_VERDICT,
              "checks": n, "rules": len(fm.RULES), "processing_time_ms": round(ms_s1, 1),
              "outputs": [f"03_outputs/{STEP}/{f}" for f in ["accounting_checks.csv", "accounting_rule_summary.csv", "row_flags.csv"]]})
assert S1_VERDICT == "PASS", "Có luật kế toán FAIL – xem bảng trên trước khi tính chỉ tiêu"

---
## BƯỚC 2 · Tính 18 chỉ tiêu tài chính

- **Một thư viện công thức duy nhất** (`finreport_metrics.METRICS`): NB2, NB3 và dashboard đều gọi lại, nên số liệu ở mọi nơi luôn khớp nhau.
- Tên chỉ tiêu trùng tên cột trong `submission_output_template.csv`.
- **Năm 2021**: các chỉ tiêu cần năm trước (tăng trưởng, ROA/ROE bình quân, DSO, Inventory Days, DPO) để trống và gắn cờ `INSUFFICIENT_HISTORY`, không tự sinh số liệu 2020.
- **Giá trị luôn đúng công thức của BTC**; nếu mẫu số bất thường (LNST ≤ 0, vốn chủ ≤ 0, LN hoạt động ≤ 0) thì vẫn tính nhưng gắn **cờ diễn giải** ở cột riêng để bước cảnh báo không hiểu sai.

In [ ]:
display(fm.metric_catalog())

In [ ]:
t0 = time.perf_counter()
RUN_S2 = new_run_id("S2")
metrics_wide, metrics_long = fm.compute_metrics(master)
ms_s2 = (time.perf_counter() - t0) * 1000

print(f"Run {RUN_S2}: {metrics_wide.shape[0]} dòng × {len(fm.METRIC_IDS)} chỉ tiêu · {ms_s2:.0f} ms")
print("\nPhân bố cờ:")
display(metrics_long["flag"].replace("", "(không cờ)").value_counts().to_frame("số ô"))

In [ ]:
# Xem bảng chỉ tiêu của một doanh nghiệp (đổi mã tùy ý)
CID = "C012"
view = metrics_wide[metrics_wide["company_id"] == CID].set_index("fiscal_year")[fm.METRIC_IDS].T
view.insert(0, "đơn vị", [fm.METRIC_BY_ID[m].unit for m in view.index])
display(view.round(2))
display(metrics_long.query("company_id == @CID and flag != ''")[["fiscal_year", "metric_id", "value", "flag"]])

In [ ]:
save_output(metrics_wide, STEP, "metrics_wide.csv")
save_output(metrics_long, STEP, "metrics_long.csv")
save_output(fm.metric_catalog(), STEP, "metric_catalog.csv")
append_audit({"run_id": RUN_S2, "step": "step2_metrics", "parent_run": RUN_S1, "verdict": "PASS",
              "metrics": len(fm.METRIC_IDS), "cells": int(metrics_long.shape[0]),
              "flags": metrics_long["flag"].replace("", "NONE").value_counts().to_dict(),
              "processing_time_ms": round(ms_s2, 1),
              "outputs": [f"03_outputs/{STEP}/{f}" for f in ["metrics_wide.csv", "metrics_long.csv", "metric_catalog.csv"]]})

---
## BƯỚC 3 · Verify & Numerical QA

| Phép kiểm | Cách làm | Chứng minh điều gì |
|---|---|---|
| **3a. Verify kép** | Tính lại 18 chỉ tiêu bằng đường thứ hai: đọc thẳng 3 file BCTC gốc bằng module `csv`, tra năm trước bằng dict (không dùng `master_panel`, không dùng pandas merge/shift) → so từng ô | Phép tính không phụ thuộc một cách cài đặt duy nhất |
| **3b. QA 30 câu mẫu** | Engine chỉ đọc `questions_sample` (loại câu, DN, năm, trường nguồn, câu chữ) → so với `answer_key_sample` theo dung sai BTC | Numerical QA Accuracy trên bộ mẫu công khai |
| **3c. Tổng quát hóa** | Tự sinh 300 câu **mới** (DN, năm, chỉ tiêu khác bộ mẫu); đáp án lấy từ đường tính 3a | Engine không học thuộc 30 câu mẫu – sẵn sàng cho câu hỏi ẩn |
| **3d. Kiểm thử đột biến** | Cố ý cài sai 1 công thức (Quick Ratio cộng thêm tồn kho) | Bộ verify thật sự bắt được lỗi công thức |
| **3e. Đối chiếu metrics mẫu BTC** | So từng ô với `financial_metrics_sample.csv` (18 company-year × 23 chỉ tiêu) | **Calculation Accuracy chính thức** |

### 3a. Verify kép

In [ ]:
t0 = time.perf_counter()
RUN_S3 = new_run_id("S3")
indep = fm.compute_metrics_independent(cfg.data_dir)
verify = fm.dual_verify(metrics_wide, indep)
cells_ok, cells_all = int(verify["agree"].sum()), int(verify["cells"].sum())
print(f"Verify kép: {cells_ok}/{cells_all} ô trùng khớp ({cells_ok/cells_all:.2%}) · chênh lệch lớn nhất {verify['max_abs_diff'].max():.2e}")
display(verify)

### 3b. Numerical QA – 30 câu mẫu của BTC

In [ ]:
questions = read_pack("questions_sample")
answer_key = read_pack("answer_key_sample")
engine = qa.QAEngine(master, metrics_wide, metrics_long)

answers = engine.answer_all(questions)            # engine KHÔNG đọc answer_key
grading = qa.grade(answers, answer_key)           # chỉ dùng answer_key để chấm
QA_ACC = grading["correct"].mean()

print(f"Numerical QA Accuracy: {grading['correct'].sum()}/{len(grading)} = {QA_ACC:.1%}  (mục tiêu BTC ≥ 85%)")
display(grading.groupby(["question_type"]).agg(số_câu=("correct", "size"), đúng=("correct", "sum")))

In [ ]:
# Bảng đối chiếu từng câu: đáp án của hệ thống vs đáp án BTC
display(grading[["question_id", "question_type", "our_normalized", "reference_normalized", "tolerance", "correct", "note"]])

In [ ]:
# Xem lời giải thích + bằng chứng của vài câu
display(answers.loc[answers["question_id"].isin(["Q009", "Q015", "Q021", "Q023", "Q025"]),
                    ["question_id", "answer", "explanation", "evidence_source_fields", "confidence"]])

In [ ]:
# Calculation Accuracy trên các câu tính toán số học (loại trừ tra cứu trực tiếp / xếp hạng / Có-Không / văn bản)
calc_types = ["RATIO_CALCULATION", "GROWTH_CALCULATION", "CHANGE_CALCULATION", "CASHFLOW_CALCULATION",
              "WORKING_CAPITAL_METRIC", "LIQUIDITY_CALCULATION"]
calc = grading[grading["question_type"].isin(calc_types)]
CALC_ACC = calc["correct"].mean()          # tạm thời; mục 3e sẽ thay bằng đối chiếu metrics mẫu nếu có
CALC_ACC_QA = CALC_ACC
print(f"Độ đúng các câu tính toán trong answer key: {calc['correct'].sum()}/{len(calc)} = {CALC_ACC:.1%}")

### 3c. Kiểm thử tổng quát hóa – 300 câu mới (mô phỏng câu hỏi ẩn)

In [ ]:
gen_q, gen_key = qa.make_variant_suite(indep, master, n=300, seed=2026)
gen_answers = engine.answer_all(gen_q)
gen_grading = qa.grade(gen_answers, gen_key)
GEN_ACC = gen_grading["correct"].mean()
print(f"Tổng quát hóa: {gen_grading['correct'].sum()}/{len(gen_grading)} = {GEN_ACC:.1%}")
display(gen_grading.groupby("question_type").agg(số_câu=("correct", "size"), đúng=("correct", "sum")))
display(gen_q.sample(5, random_state=1)[["question_id", "question_type", "question_text_vi"]])
wrong = gen_grading[~gen_grading["correct"]]
if len(wrong):
    display(wrong)

### 3d. Kiểm thử đột biến – cố ý cài sai công thức

In [ ]:
# Đột biến: Quick Ratio tính nhầm cả tồn kho (lỗi phổ biến). Chỉ sửa trên bản sao danh sách công thức.
mutant = [dataclasses.replace(m, func=lambda d, p: (d[fm.C["cash"]] + d[fm.C["ar"]] + d[fm.C["inv"]]) / d[fm.C["tcl"]])
          if m.metric_id == "quick_ratio_x" else m for m in fm.METRICS]
mut_wide, mut_long = fm.compute_metrics(master, mutant)
mut_verify = fm.dual_verify(mut_wide, indep)
mut_grading = qa.grade(qa.QAEngine(master, mut_wide, mut_long).answer_all(questions), answer_key)

caught_verify = (mut_verify["status"] == "FAIL").any()
caught_qa = not mut_grading.loc[mut_grading["question_id"] == "Q006", "correct"].iloc[0]
print(f"Verify kép phát hiện đột biến: {'CÓ' if caught_verify else 'KHÔNG'}")
display(mut_verify[mut_verify["status"] == "FAIL"])
print(f"QA phát hiện (câu Quick Ratio Q006 bị chấm sai): {'CÓ' if caught_qa else 'KHÔNG'}")
print("Công thức gốc trong fm.METRICS không bị thay đổi:",
      fm.METRIC_BY_ID["quick_ratio_x"].formula)

### 3e. Calculation Accuracy – đối chiếu `financial_metrics_sample.csv` của BTC
File mẫu có 18 company-year × 23 chỉ tiêu: 18 chỉ tiêu thuộc template nộp + 5 chỉ tiêu bổ sung
(tổng nợ vay, CFO, tăng trưởng phải thu / tồn kho / nợ vay). Hai thước đo:
- **Khớp tuyệt đối**: giá trị của hệ thống, làm tròn đến đúng số chữ số BTC ghi, bằng giá trị BTC.
- **Trong dung sai BTC**: |lệch| ≤ dung sai BTC dùng trong answer key (0,05 với %, 0,01 với hệ số, 0,2 với ngày, 0 với tiền).

In [ ]:
supp = fm.compute_supplementary(master)
ms_path, _ = locate_pack_file("financial_metrics_sample")
if ms_path is not None:
    mcmp = fm.compare_with_metrics_sample(metrics_wide, supp, ms_path)
    tpl = mcmp[mcmp["in_template"]]
    CALC_ACC = tpl["within_btc_tolerance"].mean()
    CALC_EXACT = tpl["exact_at_btc_precision"].mean()
    ALL_EXACT = mcmp["exact_at_btc_precision"].mean()
    CALC_SOURCE = "financial_metrics_sample.csv"
    print(f"Calculation Accuracy (18 chỉ tiêu template, {len(tpl)} ô):")
    print(f"   trong dung sai BTC : {tpl['within_btc_tolerance'].sum()}/{len(tpl)} = {CALC_ACC:.2%}   (mục tiêu ≥ 95%)")
    print(f"   khớp tuyệt đối     : {tpl['exact_at_btc_precision'].sum()}/{len(tpl)} = {CALC_EXACT:.2%}")
    print(f"Toàn bộ 23 chỉ tiêu ({len(mcmp)} ô), khớp tuyệt đối: {mcmp['exact_at_btc_precision'].sum()}/{len(mcmp)} = {ALL_EXACT:.2%}")
    display(mcmp.groupby("metric_id").agg(số_ô=("exact_at_btc_precision", "size"),
                                          khớp_tuyệt_đối=("exact_at_btc_precision", "sum"),
                                          trong_dung_sai=("within_btc_tolerance", "sum"),
                                          lệch_lớn_nhất=("abs_diff", "max")).round(6))
    off = mcmp[~mcmp["exact_at_btc_precision"]]
    print("\nÔ không khớp tuyệt đối:" if len(off) else "\nMọi ô khớp tuyệt đối.")
    if len(off):
        display(off[["company_id", "fiscal_year", "metric_id", "ours", "btc", "btc_decimals", "abs_diff", "within_btc_tolerance"]])
        print("Diễn giải: lệch chỉ ở chữ số làm tròn cuối cùng, nhỏ hơn nhiều so với dung sai BTC → công thức đúng.")
    save_output(mcmp, STEP, "metrics_sample_comparison.csv")
    save_output(supp, STEP, "metrics_supplementary.csv")
else:
    CALC_EXACT, ALL_EXACT, CALC_SOURCE = float("nan"), float("nan"), "answer_key (15 câu tính toán)"
    print("Chưa có financial_metrics_sample.csv – dùng Calculation Accuracy từ 15 câu tính toán trong answer key.")

### Lưu kết quả Bước 3

In [ ]:
ms_s3 = (time.perf_counter() - t0) * 1000
qa_template = read_pack("qa_submission_template")
qa_submission = qa.to_submission(answers, qa_template)

save_output(indep, STEP, "metrics_independent.csv")
save_output(verify, STEP, "verify_dual.csv")
save_output(answers, STEP, "qa_answers_detail.csv")
save_output(grading, STEP, "qa_grading.csv")
save_output(qa_submission, STEP, "qa_submission_draft.csv")       # đúng schema template, bản nháp (bản nộp cuối do NB3 xuất)
save_output(gen_grading, STEP, "qa_generalization_grading.csv")

KPI = {"calculation_accuracy": round(float(CALC_ACC), 4), "calculation_accuracy_source": CALC_SOURCE,
       "calculation_exact_match": None if pd.isna(CALC_EXACT) else round(float(CALC_EXACT), 4),
       "calc_questions_in_answer_key": round(float(CALC_ACC_QA), 4),
       "numerical_qa_accuracy": round(float(QA_ACC), 4),
       "dual_verify_agreement": round(cells_ok / cells_all, 4), "generalization_accuracy": round(float(GEN_ACC), 4),
       "mutation_caught_by_verify": bool(caught_verify), "mutation_caught_by_qa": bool(caught_qa),
       "qa_avg_latency_ms": round(float(answers["processing_time_ms"].mean()), 3)}
append_audit({"run_id": RUN_S3, "step": "step3_verify_qa", "parent_run": RUN_S2,
              "verdict": "PASS" if (cells_ok == cells_all and QA_ACC >= 0.85) else "REVIEW",
              "kpi": KPI, "model_name": qa.QAEngine.MODEL_NAME, "processing_time_ms": round(ms_s3, 1),
              "outputs": [f"03_outputs/{STEP}/{f}" for f in ["verify_dual.csv", "qa_answers_detail.csv", "qa_grading.csv",
                                                             "qa_submission_draft.csv", "qa_generalization_grading.csv",
                                                             "metrics_sample_comparison.csv", "metrics_supplementary.csv"]]})
pd.Series(KPI).to_frame("giá trị")

---
## Báo cáo tổng hợp `step1_3_report.md`

In [ ]:
pct = lambda x: f"{x:.1%}"
L = ["# Báo cáo NB1 – Kiểm tra kế toán, tính chỉ tiêu, verify & QA", "",
     f"- **Thời điểm:** {now_iso()}  ·  **Dữ liệu (fingerprint):** `{current_fp[:24]}…`",
     f"- **Chuỗi lần chạy:** `{PARENT_RUN}` → `{RUN_S1}` → `{RUN_S2}` → `{RUN_S3}`", "",
     "## KPI", "", "| KPI | Kết quả | Mục tiêu BTC |", "|---|---:|---:|",
     (f"| Calculation Accuracy – trong dung sai BTC (financial_metrics_sample, 18 chỉ tiêu) | {tpl['within_btc_tolerance'].sum()}/{len(tpl)} = **{pct(CALC_ACC)}** | ≥ 95% |\n"
      f"| Calculation – khớp tuyệt đối ở độ chính xác BTC ghi | {tpl['exact_at_btc_precision'].sum()}/{len(tpl)} = **{pct(CALC_EXACT)}** | – |\n"
      f"| 5 chỉ tiêu bổ sung (tổng nợ, CFO, tăng trưởng AR/tồn kho/nợ) | {mcmp[~mcmp['in_template']]['exact_at_btc_precision'].sum()}/{(~mcmp['in_template']).sum()} khớp tuyệt đối | – |"
      if CALC_SOURCE.startswith("financial") else
      f"| Calculation Accuracy (câu tính toán trong answer key) | {calc['correct'].sum()}/{len(calc)} = **{pct(CALC_ACC)}** | ≥ 95% |"),
     f"| Câu tính toán trong answer key | {calc['correct'].sum()}/{len(calc)} = **{pct(CALC_ACC_QA)}** | – |",
     f"| Numerical QA Accuracy (30 câu mẫu) | {grading['correct'].sum()}/{len(grading)} = **{pct(QA_ACC)}** | ≥ 85% |",
     f"| Verify kép (2 cách tính độc lập) | {cells_ok}/{cells_all} ô = **{pct(cells_ok/cells_all)}** | – |",
     f"| Tổng quát hóa (300 câu tự sinh) | {gen_grading['correct'].sum()}/{len(gen_grading)} = **{pct(GEN_ACC)}** | – |",
     f"| Kiểm thử đột biến bị phát hiện | verify: **{'Có' if caught_verify else 'Không'}** · QA: **{'Có' if caught_qa else 'Không'}** | – |",
     f"| Thời gian trả lời trung bình / câu | {answers['processing_time_ms'].mean():.2f} ms | – |", "",
     "## Bước 1 – Kiểm tra logic kế toán", "",
     f"{len(fm.RULES)} luật × 60 company-year: **{n.get('PASS',0)} PASS · {n.get('FAIL',0)} FAIL · {n.get('N/A',0)} N/A** "
     "(N/A = luật liên kết hai năm không áp dụng cho 2021 vì không có dữ liệu 2020).", "",
     "| Luật | Mô tả | PASS | FAIL | N/A |", "|---|---|---:|---:|---:|"]
L += [f"| {r['rule_id']} | {r['description']} | {r['PASS']} | {r['FAIL']} | {r['N/A']} |" for _, r in acc_summary.iterrows()]
sens = flags[flags["row_flags"].str.contains("NET_LOSS|NEGATIVE_CFO|OPERATING_LOSS|OP_BELOW_INTEREST|NEGATIVE_EQUITY")]
L += ["", "**Dòng cần diễn giải thận trọng:** " + "; ".join(f"{r.company_id}/{r.fiscal_year} ({r.row_flags})" for r in sens.itertuples()), "",
      "## Bước 2 – Chỉ tiêu", "",
      f"{len(fm.METRIC_IDS)} chỉ tiêu × 60 dòng. Cờ: " +
      ", ".join(f"{k or 'không cờ'}={v}" for k, v in metrics_long['flag'].value_counts().items()) + ".", "",
      "| Chỉ tiêu | Công thức | Cần năm trước |", "|---|---|:---:|"]
L += [f"| {m.metric_id} | {m.formula} | {'✓' if m.needs_prev else ''} |" for m in fm.METRICS]
L += ["", "## Bước 3 – QA từng câu", "", "| Câu | Loại | Hệ thống | BTC | Đúng |", "|---|---|---|---|:---:|"]
L += [f"| {r.question_id} | {r.question_type} | {str(r.our_normalized)[:40]} | {str(r.reference_normalized)[:40]} | {'✓' if r.correct else '✗'} |"
      for r in grading.itertuples()]
rep = cfg.step_dir(STEP) / "step1_3_report.md"
rep.write_text("\n".join(L), encoding="utf-8")
print("Đã ghi:", rep)
print("\nCác file trong", cfg.step_dir(STEP))
for f in sorted(cfg.step_dir(STEP).iterdir()):
    print("  ", f.name)

In [ ]:
display(read_audit().tail(4)[["timestamp", "run_id", "step", "verdict", "processing_time_ms"]])